# 02 — GRPO Reinforcement Learning

This notebook applies **Group Relative Policy Optimization (GRPO)** on top of
the SFT-tuned model to improve mathematical reasoning.

**Pipeline:**
1. Merge the SFT LoRA adapter into the base model
2. Load the merged model with Unsloth + vLLM for fast generation
3. Attach a fresh LoRA adapter for RL training
4. Train with a custom reward function (format + correctness)

**Requirements:** Google Colab with A100 GPU (High-RAM recommended).

## 1. Setup

In [ ]:
# Clone the repo (replace with your GitHub URL)
!git clone https://github.com/YOUR_USERNAME/math-rl-tuning.git
%cd math-rl-tuning

# Step 1: Install Unsloth first (it pins TRL to a compatible version)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --quiet

# Step 2: Pin vLLM to the version TRL expects (critical — wrong version causes ImportError)
!pip install "vllm==0.10.2" --quiet

# Step 3: Install our package and remaining dependencies
!pip install -e . --quiet
!pip install bitsandbytes latex2sympy2 --quiet

## 2. Configuration

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
from math_rl_tuning.config import load_config
from math_rl_tuning.utils import setup_hf_token, setup_wandb, mount_google_drive

cfg = load_config()

# --- Authentication ---
setup_hf_token()
setup_wandb(cfg.grpo_training.report_to and "math-rl-grpo")

# Mount Google Drive (to load SFT adapter and save RL model)
mount_google_drive()

## 3. (Optional) Customize Config

In [ ]:
# Point to your SFT adapter (from notebook 01 or Google Drive)
# If you saved to Drive in notebook 01, it will be at:
SFT_ADAPTER_PATH = cfg.paths.sft_output_dir
# Or from Drive:
# SFT_ADAPTER_PATH = "/content/drive/MyDrive/math-rl-tuning/sft"

# Adjust GRPO hyperparameters if desired
# cfg.grpo_training.learning_rate = 3e-5
# cfg.grpo_training.grpo_sample_size = 1200
# cfg.grpo_training.num_generations = 16

# Adjust reward weights
# cfg.rewards.correct_bonus = 1.5
# cfg.rewards.length_threshold = 300

## 4. Run GRPO Training

In [ ]:
from math_rl_tuning.grpo_trainer import run_grpo_training

trainer, model, tokenizer = run_grpo_training(
    cfg,
    sft_adapter_path=SFT_ADAPTER_PATH,
    save_to_drive=True,
)

## 5. Quick Sanity Check

In [ ]:
from math_rl_tuning.inference import generate

questions = [
    "What is 15% of 240?",
    "Solve for x: 3x + 7 = 22",
    "A rectangle has length 12 cm and width 5 cm. What is its area?",
]

for q in questions:
    print(f"Q: {q}")
    response = generate(q, model, tokenizer)
    print(f"A: {response[:300]}")
    print("-" * 40)

## 6. Cleanup

In [ ]:
from math_rl_tuning.utils import clean_memory

del model, trainer
clean_memory()